<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Import Libraries</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Import Libraries
    </h1>
</div>


In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from config.spark_config import SparkConfig
from config.io_config import *
from app.platform_app import PlatformApp
from transformation.gold_loader import *
from utils.logger import LoggerConfig
from transformation.bronze_ingestion import *
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from utils.data_quality import *
from utils.data_cleaning import *
from utils.utils import *

<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Set up</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Set up
    </h1>
</div>


In [2]:
logger = LoggerConfig().setup_logger(log_dir=None)
spark = SparkConfig.create_spark(app_name="FMCG Domain", logger=logger, use_databricks=True)
app = PlatformApp(spark=spark, logger=logger, catalog_name="fmcg_domain")

2026-03-20 22:15:53 | INFO     | Logging configured: level=DEBUG, format=colored
2026-03-20 22:15:54 | INFO     | Connected to Databricks via Spark Connect.
2026-03-20 22:15:54 | INFO     | Initializing Data Platform...
2026-03-20 22:15:54 | INFO     | Spark session initialized


<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Bronze</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Bronze
    </h1>
</div>


In [3]:
# Bronze Ingestion
logger.info("Ingesting data into Bronze layer...")
upload_file_to_bronze(spark=spark, logger=logger, entity="gross_price",
                      path_data=S3_BASE_PATH, path_cp=CP_PATH_BRONZE,
                      name_catalog=app.catalog_name)
logger.info("Bronze ingestion completed.")
logger.info("*" * 80)

2026-03-20 22:15:54 | INFO     | Ingesting data into Bronze layer...
2026-03-20 22:15:54 | INFO     | Starting Bronze ingestion for entity: Gross_Price
2026-03-20 22:15:54 | INFO     | Uploading file: s3://00-sportsbar-dp/gross_price
2026-03-20 22:16:15 | INFO     | Gross_Price: Schema inferred successfully
2026-03-20 22:16:28 | INFO     | Completed Bronze ingestion for entity: Gross_Price
+----------+----------+-----------+-----------------------+---------------+---------+----------------------+
|product_id|month     |gross_price|read_timestamp         |file_name      |file_size|file_modification_time|
+----------+----------+-----------+-----------------------+---------------+---------+----------------------+
|25891101  |2025/07/01|-84        |2026-03-20 13:25:17.294|gross_price.csv|2741     |2026-03-05 13:07:43   |
|25891101  |01/08/2025|unknown    |2026-03-20 13:25:17.294|gross_price.csv|2741     |2026-03-05 13:07:43   |
|25891101  |2025/09/01|84         |2026-03-20 13:25:17.294|gro

<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Silver</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Silver
    </h1>
</div>


In [4]:
df_silver = spark.sql(f"SELECT * FROM {app.catalog_name}.bronze.gross_price")
df_silver.show(truncate=False)

+----------+----------+-------------+-----------------------+---------------+---------+----------------------+
|product_id|month     |gross_price  |read_timestamp         |file_name      |file_size|file_modification_time|
+----------+----------+-------------+-----------------------+---------------+---------+----------------------+
|25891101  |2025/07/01|-84          |2026-03-20 13:25:17.294|gross_price.csv|2741     |2026-03-05 13:07:43   |
|25891101  |01/08/2025|unknown      |2026-03-20 13:25:17.294|gross_price.csv|2741     |2026-03-05 13:07:43   |
|25891101  |2025/09/01|84           |2026-03-20 13:25:17.294|gross_price.csv|2741     |2026-03-05 13:07:43   |
|25891101  |2025-10-01|83           |2026-03-20 13:25:17.294|gross_price.csv|2741     |2026-03-05 13:07:43   |
|25891101  |2025-11-01|83           |2026-03-20 13:25:17.294|gross_price.csv|2741     |2026-03-05 13:07:43   |
|88888888  |2025-12-01|-83          |2026-03-20 13:25:17.294|gross_price.csv|2741     |2026-03-05 13:07:43   |
|

## Transformations

### Duplicates

In [5]:
df_duplicates = check_duplicate(df=df_silver, logger=logger)

2026-03-20 22:16:41 | INFO     | No duplicate rows found out of 108 total rows.


### Check NULL

In [6]:
# check null
check_null(df = df_silver, logger=logger)

2026-03-20 22:16:43 | INFO     | No missing values detected in 108 rows.


### Trim spaces in customer name

In [7]:
# remove those trim values
df_silver = clean_dataframe(df=df_silver)
df_silver.show(truncate=False)

+----------+----------+-------------+-----------------------+---------------+---------+----------------------+
|product_id|month     |gross_price  |read_timestamp         |file_name      |file_size|file_modification_time|
+----------+----------+-------------+-----------------------+---------------+---------+----------------------+
|25891101  |2025/07/01|-84          |2026-03-20 13:25:17.294|gross_price.csv|2741     |2026-03-05 13:07:43   |
|25891101  |01/08/2025|unknown      |2026-03-20 13:25:17.294|gross_price.csv|2741     |2026-03-05 13:07:43   |
|25891101  |2025/09/01|84           |2026-03-20 13:25:17.294|gross_price.csv|2741     |2026-03-05 13:07:43   |
|25891101  |2025-10-01|83           |2026-03-20 13:25:17.294|gross_price.csv|2741     |2026-03-05 13:07:43   |
|25891101  |2025-11-01|83           |2026-03-20 13:25:17.294|gross_price.csv|2741     |2026-03-05 13:07:43   |
|88888888  |2025-12-01|-83          |2026-03-20 13:25:17.294|gross_price.csv|2741     |2026-03-05 13:07:43   |
|

### Normalise `month` field

In [8]:
df_silver.select("month").distinct().show()

+----------+
|     month|
+----------+
|2025/07/01|
|01/08/2025|
|2025/09/01|
|2025-10-01|
|2025-11-01|
|2025-12-01|
|2025-07-01|
|2025-08-01|
|2025-09-01|
|2025/11/01|
|2025/08/01|
|01-09-2025|
|2025/10/01|
|01/12/2025|
|01/09/2025|
|01-12-2025|
|01-08-2025|
|01/10/2025|
+----------+



In [9]:
# Parse `month` from multiple possible formats
date_formats = ["yyyy/MM/dd", "dd/MM/yyyy", "yyyy-MM-dd", "dd-MM-yyyy"]

df_silver = df_silver.withColumn(
    "month",
    F.coalesce(
        F.expr("try_cast(month as date)"),
        F.expr("try_to_date(month,'dd/MM/yyyy')"),
        F.expr("try_to_date(month,'yyyy/MM/dd')"),
        F.expr("try_to_date(month,'yyyy-MM-dd')"),
        F.expr("try_to_date(month,'dd-MM-yyyy')")
    )
)


In [10]:
df_silver.select("month").distinct().show()

+----------+
|     month|
+----------+
|2025-07-01|
|2025-08-01|
|2025-09-01|
|2025-10-01|
|2025-11-01|
|2025-12-01|
+----------+



### Handling `gross_price`

In [11]:
df_silver.show(truncate=False)

+----------+----------+-------------+-----------------------+---------------+---------+----------------------+
|product_id|month     |gross_price  |read_timestamp         |file_name      |file_size|file_modification_time|
+----------+----------+-------------+-----------------------+---------------+---------+----------------------+
|25891101  |2025-07-01|-84          |2026-03-20 13:25:17.294|gross_price.csv|2741     |2026-03-05 13:07:43   |
|25891101  |2025-08-01|unknown      |2026-03-20 13:25:17.294|gross_price.csv|2741     |2026-03-05 13:07:43   |
|25891101  |2025-09-01|84           |2026-03-20 13:25:17.294|gross_price.csv|2741     |2026-03-05 13:07:43   |
|25891101  |2025-10-01|83           |2026-03-20 13:25:17.294|gross_price.csv|2741     |2026-03-05 13:07:43   |
|25891101  |2025-11-01|83           |2026-03-20 13:25:17.294|gross_price.csv|2741     |2026-03-05 13:07:43   |
|88888888  |2025-12-01|-83          |2026-03-20 13:25:17.294|gross_price.csv|2741     |2026-03-05 13:07:43   |
|

In [12]:
# We are validating the gross_price column, converting only valid numeric values to double, 
# fixing negative prices by making them positive, and replacing all non-numeric values with 0
df_silver = df_silver.withColumn(
    "gross_price",
    F.coalesce(F.abs(F.expr("try_cast(gross_price as double)")), F.lit(0))
)

df_silver.show(truncate=False)

+----------+----------+-----------+-----------------------+---------------+---------+----------------------+
|product_id|month     |gross_price|read_timestamp         |file_name      |file_size|file_modification_time|
+----------+----------+-----------+-----------------------+---------------+---------+----------------------+
|25891101  |2025-07-01|84.0       |2026-03-20 13:25:17.294|gross_price.csv|2741     |2026-03-05 13:07:43   |
|25891101  |2025-08-01|0.0        |2026-03-20 13:25:17.294|gross_price.csv|2741     |2026-03-05 13:07:43   |
|25891101  |2025-09-01|84.0       |2026-03-20 13:25:17.294|gross_price.csv|2741     |2026-03-05 13:07:43   |
|25891101  |2025-10-01|83.0       |2026-03-20 13:25:17.294|gross_price.csv|2741     |2026-03-05 13:07:43   |
|25891101  |2025-11-01|83.0       |2026-03-20 13:25:17.294|gross_price.csv|2741     |2026-03-05 13:07:43   |
|88888888  |2025-12-01|83.0       |2026-03-20 13:25:17.294|gross_price.csv|2741     |2026-03-05 13:07:43   |
|25891102  |2025-07

In [13]:
df_silver.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- month: date (nullable = true)
 |-- gross_price: double (nullable = false)
 |-- read_timestamp: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)
 |-- file_modification_time: timestamp (nullable = true)



### Enrich the silver dataset

In [14]:
# We enrich the silver dataset by performing an inner join with the products table to fetch 
# the correct product_code for each product_id.

df_products = spark.table("fmcg_domain.silver.dim_products") 
df_joined = df_silver.join(df_products.select("product_id", "product_code", "product"), on="product_id", how="inner")
df_joined = df_joined.select("product_id", "product_code", "product", "month", "gross_price", 
                             "read_timestamp", "file_name", "file_size")

df_joined.show(truncate=False)

+----------+----------------------------------------------------------------+-----------------------------------------+----------+-----------+-----------------------+---------------+---------+
|product_id|product_code                                                    |product                                  |month     |gross_price|read_timestamp         |file_name      |file_size|
+----------+----------------------------------------------------------------+-----------------------------------------+----------+-----------+-----------------------+---------------+---------+
|25891101  |e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843|SportsBar Energy Bar Choco Fudge (60g)   |2025-07-01|84.0       |2026-03-20 13:25:17.294|gross_price.csv|2741     |
|25891101  |e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843|SportsBar Energy Bar Choco Fudge (60g)   |2025-08-01|0.0        |2026-03-20 13:25:17.294|gross_price.csv|2741     |
|25891101  |e91ba9d665f90254da5809b

### Transformed data to Silver Layer

In [15]:
if not spark.catalog.tableExists(SILVER_GROSS_PRICE):
    logger.info("Silver Gross Price table not found. Creating new table...")
    df_joined.write.format("delta") \
                   .option("delta.enableChangeDataFeed", "true") \
                   .option("mergeSchema", "true") \
                   .mode("overwrite") \
                   .saveAsTable(SILVER_GROSS_PRICE)
    logger.info("Silver Gross Price table created successfully")
else:
    logger.info("Silver Gross Price table exists. Performing upsert...")
    upsert(spark=spark, df=df_joined, key_cols=["product_id","month"],
           table="dim_gross_price", cdc="read_timestamp",
           name_catalog=app.catalog_name, name_schema="silver", logger=logger)
    logger.info("Upsert completed successfully")

2026-03-20 22:16:53 | INFO     | Silver Gross Price table exists. Performing upsert...
2026-03-20 22:16:53 | INFO     | Starting UPSERT into fmcg_domain.silver.dim_gross_price
2026-03-20 22:17:08 | INFO     | UPSERT completed successfully: fmcg_domain.silver.dim_gross_price
2026-03-20 22:17:08 | INFO     | Upsert completed successfully


<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Gold</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Gold
    </h1>
</div>


In [16]:
df_silver = spark.sql(f"SELECT * FROM {app.catalog_name}.silver.dim_gross_price")
# select only required columns
df_gold = df_silver.select("product_code", "month", "gross_price")
df_gold.show(truncate=False)

+----------------------------------------------------------------+----------+-----------+
|product_code                                                    |month     |gross_price|
+----------------------------------------------------------------+----------+-----------+
|451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa|2025-12-01|187.0      |
|451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa|2025-10-01|187.0      |
|451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa|2025-09-01|171.0      |
|451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa|2025-08-01|171.0      |
|451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa|2025-07-01|171.0      |
|778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6|2025-12-01|296.0      |
|778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6|2025-11-01|296.0      |
|778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6|2025-09-01|289.0      |
|778c2a7aa

In [19]:
# select only required columns
df_gold = df_gold.select("product_code", "month", "gross_price")
df_gold.show(5)

+--------------------+----------+-----------+
|        product_code|     month|gross_price|
+--------------------+----------+-----------+
|451f7167b28a25bde...|2025-12-01|      187.0|
|451f7167b28a25bde...|2025-10-01|      187.0|
|451f7167b28a25bde...|2025-09-01|      171.0|
|451f7167b28a25bde...|2025-08-01|      171.0|
|451f7167b28a25bde...|2025-07-01|      171.0|
+--------------------+----------+-----------+
only showing top 5 rows


In [ ]:
logger.info("Gold Gross Price table not found. Creating new table...")
df_gold.write.format("delta") \
                .option("delta.enableChangeDataFeed", "true") \
                .option("mergeSchema", "true") \
                .mode("overwrite") \
                .saveAsTable(GOLD_SB_GROSS_PRICE)
logger.info("Gold Gross Price table created successfully")

## Merging Data source with parent

### Get the price for each product_code (aggregated by year)

In [ ]:
# Step 1: Extract year from month column
# Mục đích: tạo partition theo từng năm cho mỗi product
df_gold_price = (
    df_gold
    .withColumn("year", F.year("month"))

    # Step 2: Flag giá = 0
    # is_zero = 1 nếu gross_price = 0
    # is_zero = 0 nếu gross_price > 0
    # dùng để ưu tiên giá khác 0 khi ranking
    .withColumn("is_zero", F.when(F.col("gross_price") == 0, 1).otherwise(0))
)

# Step 3: Define window specification
# Partition theo product_code + year
# Sort theo:
#   1. is_zero ASC  → giá khác 0 (0) sẽ đứng trước
#   2. month DESC   → tháng mới nhất đứng trước
w = (
    Window
    .partitionBy("product_code", "year")
    .orderBy(F.col("is_zero"), F.col("month").desc())
)

# Step 4: Apply row_number để rank record trong mỗi partition
# rnk = 1 là record ưu tiên nhất:
#   - giá khác 0
#   - tháng mới nhất
df_gold_latest = (
    df_gold_price
    .withColumn("rnk", F.row_number().over(w))
    # Step 5: chỉ giữ record rank = 1 tức là latest non-zero price của product trong năm
    .filter(F.col("rnk") == 1)
)

In [22]:
df_gold_latest.show(truncate=False)

+----------------------------------------------------------------+----------+-----------+----+-------+---+
|product_code                                                    |month     |gross_price|year|is_zero|rnk|
+----------------------------------------------------------------+----------+-----------+----+-------+---+
|062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379|2025-12-01|281.0      |2025|0      |1  |
|0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28|2025-10-01|100.0      |2025|0      |1  |
|102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268|2025-12-01|86.0       |2025|0      |1  |
|2e387cef1424d6e7b162b45622d4b1a788d11776e33d05cc8552f4ecd2ea1896|2025-12-01|108.0      |2025|0      |1  |
|3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345|2025-12-01|493.0      |2025|0      |1  |
|451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa|2025-12-01|187.0      |2025|0      |1  |
|716fa4e54b7894c910180276e0535d49afb2

In [23]:
## Take required cols
df_gold_latest = df_gold_latest.select("product_code", "year", "gross_price").withColumnRenamed("gross_price", "price_inr").select("product_code", "price_inr", "year")

# change year to string
df_gold_latest = df_gold_latest.withColumn("year", F.col("year").cast("string"))

df_gold_latest.show(truncate=False)

+----------------------------------------------------------------+---------+----+
|product_code                                                    |price_inr|year|
+----------------------------------------------------------------+---------+----+
|062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379|281.0    |2025|
|0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28|100.0    |2025|
|102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268|86.0     |2025|
|2e387cef1424d6e7b162b45622d4b1a788d11776e33d05cc8552f4ecd2ea1896|108.0    |2025|
|3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345|493.0    |2025|
|451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa|187.0    |2025|
|716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742|440.0    |2025|
|778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6|296.0    |2025|
|77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9|50.0     |2025|
|889c67757ece9c9

In [24]:
delta_table = DeltaTable.forName(spark, GOLD_GROSS_PRICE)

delta_table.alias("target").merge(
    source=df_gold_latest.alias("source"),
    condition="target.product_code = source.product_code"
).whenMatchedUpdate(
    set={
        "price_inr": "source.price_inr",
        "year": "source.year"
    }
).whenNotMatchedInsert(
    values={
        "product_code": "source.product_code",
        "price_inr": "source.price_inr",
        "year": "source.year"
    }
).execute()

,num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,17,0,0,17
